In [1]:
import json
import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import login
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch
import re
from tqdm import tqdm

# check if gpu is available
device = 0 if torch.cuda.is_available() else -1
print("Using device:", "GPU" if device == 0 else "CPU")

load_dotenv()  
hf_token = os.getenv("HF_TOKEN")
login(token=hf_token)

c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [ ]:
MODELS = [
    'meta-llama/Llama-3.2-3B-Instruct',
    #'Qwen/Qwen2.5-3B-Instruct',
]

def load_model(model_name: str):
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="cuda",
        torch_dtype="auto",
        trust_remote_code=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    return pipeline("text-generation", model=model, tokenizer=tokenizer), model, tokenizer

def destroy_model(model, tokenizer):
    if model is not None:
        del model
    
    if tokenizer is not None:
        del tokenizer

    torch.cuda.empty_cache()

In [ ]:
import re
from typing import Optional

def clean_output(text: str, output_type: str) -> Optional[str]:
    """
    Estrae in modo robusto il contenuto tra <REASONING>...</REASONING> o <ANSWER>...</ANSWER>.
    Gestisce tag mancanti/troncati e differenze di maiuscole/minuscole.
    Restituisce None se non trova nulla.
    """
    if not text:
        return None

    flags = re.DOTALL | re.IGNORECASE

    if output_type == "reasoning":
        # 1) caso ideale: <REASONING>... </REASONING>
        m = re.search(r'<REASONING\s*>(.*?)</REASONING\s*>', text, flags)
        if m:
            return m.group(1).strip()

        # 2) se non c'è chiusura, estrai da <REASONING> fino al tag <ANSWER> (se presente)
        m = re.search(r'<REASONING\s*>(.*?)(?=<ANSWER\b)', text, flags)
        if m:
            return m.group(1).strip()

        # 3) se c'è solo <REASONING> e poi la fine del testo (troncato), prendi tutto il resto
        m = re.search(r'<REASONING\s*>(.*)$', text, flags)
        if m:
            return m.group(1).strip()

        return None

    elif output_type == "answer":
        # 1) caso ideale: <ANSWER>... </ANSWER>
        m = re.search(r'<ANSWER\s*>(.*?)</ANSWER\s*>', text, flags)
        if m:
            answer_text = m.group(1).strip()
            # Cerca solo 0 o 1 nella risposta
            digit_match = re.search(r'[01]', answer_text)
            if digit_match:
                return digit_match.group(0)
            return answer_text

        # 2) se la chiusura manca, cerca una cifra 0/1 subito dopo <ANSWER>
        m = re.search(r'<ANSWER\s*>(\s*[01])\b', text, flags)
        if m:
            return m.group(1).strip()

        # 3) se c'è solo <ANSWER> e poi testo troncato -> prendi il resto e cerca 0/1 dentro
        m = re.search(r'<ANSWER\s*>(.*)$', text, flags)
        if m:
            candidate = m.group(1).strip()
            d = re.search(r'([01])', candidate)
            if d:
                return d.group(1)
            return candidate  # fallback: ritorna quello che c'è

        # 4) fallback ulteriore: se non ci sono tag ma l'output finisce con una riga contenente solo 0 o 1
        m = re.search(r'\n\s*([01])\s*$', text, flags)
        if m:
            return m.group(1)

        return None

    else:
        raise ValueError("Invalid output_type. Must be 'answer' or 'reasoning'.")


def generate_output(pipe, tokenizer, input_text: str, temperature: float = 0.1):
    prompt = f"""        
        Classify whether the use of irony in the response to the user utterance between the tag <USER_UTTERANCE>...</USER_UTTERANCE> is appropriate. Output must be 0 (inappropriate) or 1 (appropriate).
        
        reason step by step, and then provide your final answer between the tags <ANSWER></ANSWER>.
        Provide your reasoning between the tags <REASONING></REASONING>.

        <USER_UTTERANCE>{input_text}</USER_UTTERANCE>
        <REASONING>
    """

    output = pipe(
        prompt,
        max_new_tokens=200,
        do_sample=temperature > 0,
        temperature=temperature,
        return_full_text=False,
        pad_token_id=tokenizer.eos_token_id
    )

    # Ora output[0]['generated_text'] contiene SOLO la risposta generata, non il prompt
    model_output = output[0]['generated_text'].strip()    
    reasoning = clean_output(f"<REASONING>{model_output}", "reasoning")
    answer = clean_output(model_output, "answer")

    return prompt, model_output, reasoning, answer

In [ ]:
def calculate_aggregated_baseline(dataset_df: pd.DataFrame):
    output_df = dataset_df.copy()
    
    for model_name in MODELS:
        print(f"Processing model: {model_name}")
        pipe, model, tokenizer = load_model(model_name)
        print("Model and tokenizer loaded.")

        # Prepare output columns
        output_df[f'prompt'] = None
        output_df[f'model_output'] = None
        output_df[f'model_reasoning'] = None
        output_df[f'model_answer'] = None

        # Add tqdm progress bar
        output_df_rows = []
        for idx, row in tqdm(dataset_df.iterrows(), total=len(dataset_df), desc=f"Running {model_name}"):
            answer = None
            while answer not in ['0', '1']:
                prompt, model_output, reasoning, answer = generate_output(pipe, tokenizer, row['post'])

            # Creiamo la riga aggiornata
            enriched_row = row.copy()
            enriched_row[f'prompt'] = prompt
            enriched_row[f'model_output'] = model_output
            enriched_row[f'model_reasoning'] = reasoning
            enriched_row[f'model_answer'] = answer

            output_df_rows.append(enriched_row)

            # save as json with indent=2, ensure_ascii=False
            output_df_partial = pd.DataFrame(output_df_rows)
            output_df_partial.to_json(f'test2_baseline_{model_name.replace("/", "_")}.json', orient='records', indent=2, force_ascii=False)

        destroy_model(model, tokenizer)
        print("Model and tokenizer destroyed.")

In [ ]:
# check if exists and open unified corpus in 1_datasets_merge\output\unified_corpus.jsonù
file_path = '../1_datasets_merge/output/unified_corpus.json'
try:
    with open(file_path, 'r', encoding='utf-8') as f:
        unified_corpus = json.load(f)
    print(f"Loaded unified corpus with {len(unified_corpus)} entries")
    df = pd.DataFrame(unified_corpus)
except Exception as e:
    print(f"Error loading unified corpus: {e}")
    raise e

# get only row with language == 'en' and dataset == 'MultiPICo'
df_en_multipico = df[(df['language'] == 'en') & (df['dataset_name'] == 'MultiPICo')]
# show number of entries
print(f"Filtered MultiPICo English entries: {len(df_en_multipico)}")

# get only 100 entries for testing
df_en_multipico = df_en_multipico.head(100)
print(f"Using {len(df_en_multipico)} entries for testing")

calculate_aggregated_baseline(df_en_multipico)